In [337]:
import heapq
import numpy as np
import pandas as pd
import plotly.express as px
import collections
import itertools
import tqdm
from copy import deepcopy

In [338]:
def normalize_priors(priors):
    if np.sum(priors) == 0:
        return np.ones_like(priors)
    return priors / np.sum(priors)

def best_response_vectorized(X, thresholds, priors, c):
    posteriors = normalize_priors(priors).flatten()
    utilities_expected = np.array([
        np.dot(posteriors, thresholds[j] >= thresholds)
        for j in range(len(thresholds))
    ])

    pass_matrix = X[:, None] >= thresholds[None, :]
    utility_stay = pass_matrix @ posteriors

    X_col = X[:, None]
    thresholds_row = thresholds[None, :]

    feasible = thresholds_row > X_col
    utility_jump = utilities_expected[None, :] - c * np.abs(thresholds_row - X_col)
    utility_jump[~feasible] = -np.inf

    utilities = np.concatenate([utility_stay[:, None], utility_jump], axis=1)
    best_idx = np.argmax(utilities + np.array([(utilities.shape[1] - i)*1e-6 for i in range(utilities.shape[1])]), axis=1)
    X_p = np.where(best_idx == 0, X, thresholds[best_idx - 1])
    return X_p

def accuracy_loss_vectorized(X, X_p, thresholds, priors, threshold_true):
    Y_true = (X >= threshold_true).astype(float)
    Y_p = (X_p[:, None] >= thresholds[None, :]).astype(float)
    losses = np.abs(Y_true[:, None] - Y_p).mean(axis=0)
    posteriors = normalize_priors(priors)
    return np.dot(losses, posteriors)

def evaluate_partition(X, partition, thresholds, priors, threshold_true, c):
    thresholds_p = thresholds[partition]
    priors_p     = priors[partition]
    X_p = best_response_vectorized(X, thresholds_p, priors_p, c)
    return accuracy_loss_vectorized(X, X_p, thresholds_p, priors_p, threshold_true)

def evaluate_system(X, partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.0
    for partition in partitions:
        acc_loss_p = evaluate_partition(X, partition, thresholds, priors, threshold_true, c)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    return acc_loss

In [387]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return
    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
    indices = list(range(len(thresholds)))
    best_partition, best_loss = None, np.inf
    for partitions in set_partitions(indices):
        acc_loss = evaluate_system(X, partitions, thresholds, priors, threshold_true, c)
        if acc_loss < best_loss:
            best_loss = acc_loss
            best_partition = partitions
    return best_partition

def find_partitions_greedy_agglomerative(X, thresholds, priors, threshold_true, c, eps=1e-9):
    P = {}
    next_id = 0
    for i in range(len(priors)):
        P[next_id] = [i]
        next_id += 1

    X_eps = np.arange(-1/c, 0, 1e-3).round(5)

    pq = []
    for a_id, b_id in itertools.combinations(P.keys(), 2):
        a, b = P[a_id], P[b_id]
        ab = sorted(a+b)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c) * np.sum(priors[ab])
        acc_loss_a  = evaluate_partition(X, a, thresholds, priors, threshold_true, c)  * np.sum(priors[a])
        acc_loss_b  = evaluate_partition(X, b, thresholds, priors, threshold_true, c)  * np.sum(priors[b])
        acc_loss_ab += evaluate_partition(X_eps, ab, thresholds, priors, threshold_true, c) * np.sum(priors[ab]) * eps
        acc_loss_a  += evaluate_partition(X_eps, a,  thresholds, priors, threshold_true, c) * np.sum(priors[a])  * eps
        acc_loss_b  += evaluate_partition(X_eps, b,  thresholds, priors, threshold_true, c) * np.sum(priors[b])  * eps
        gain = -(acc_loss_a + acc_loss_b - acc_loss_ab)
        heapq.heappush(pq, (gain, (a_id, b_id)))

    while pq:
        gain, (a_id, b_id) = heapq.heappop(pq)
        if a_id not in P or b_id not in P:
            continue
        a, b = P[a_id], P[b_id]
        ab = sorted(a + b)

        acc_loss_a  = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
        acc_loss_b  = evaluate_partition(X, b, thresholds, priors, threshold_true, c)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)
        acc_loss_a  += evaluate_partition(X_eps, a,  thresholds, priors, threshold_true, c) * eps
        acc_loss_b  += evaluate_partition(X_eps, b,  thresholds, priors, threshold_true, c) * eps
        acc_loss_ab += evaluate_partition(X_eps, ab, thresholds, priors, threshold_true, c) * eps

        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs - rhs > -1e-9:
            del P[a_id]
            del P[b_id]
            pq = [(g, (x, y)) for g, (x, y) in pq if x not in {a_id, b_id} and y not in {a_id, b_id}]
            heapq.heapify(pq)

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for p_id in P:
                if p_id == new_id:
                    continue
                p = P[p_id]
                merged = sorted(ab + p)
                acc_loss_merged = evaluate_partition(X, merged, thresholds, priors, threshold_true, c) * np.sum(priors[merged])
                acc_loss_p      = evaluate_partition(X, p,      thresholds, priors, threshold_true, c) * np.sum(priors[p])
                acc_loss_ab_new = evaluate_partition(X, ab,     thresholds, priors, threshold_true, c) * np.sum(priors[ab])
                acc_loss_merged += evaluate_partition(X_eps, merged, thresholds, priors, threshold_true, c) * np.sum(priors[merged]) * eps
                acc_loss_p      += evaluate_partition(X_eps, p,      thresholds, priors, threshold_true, c) * np.sum(priors[p])      * eps
                acc_loss_ab_new += evaluate_partition(X_eps, ab,     thresholds, priors, threshold_true, c) * np.sum(priors[ab])     * eps
                gain = -(acc_loss_p + acc_loss_ab_new - acc_loss_merged)
                heapq.heappush(pq, (gain, (new_id, p_id)))
    return list(P.values())

def approximation_ratio(loss_optimal, loss_greedy, rtype="m"):
    if rtype in ["a", "add", "additive"]:
        return loss_greedy - loss_optimal
    elif rtype in ["m", "mult", "multiplicative"]:
        if loss_optimal == 0:
            return np.nan
        return loss_greedy / loss_optimal

In [353]:
def split_partition(partition, full_split=False):
    idx_large = 0
    block_large = partition[idx_large]
    block_others = [partition[i] for i in range(len(partition)) if i != idx_large]
    result = []
    for part in itertools.combinations(block_large, len(block_large)-1):
        A = list(part)
        B = [x for x in block_large if x not in A]
        result.append([A] + [B] + block_others)

        if full_split:
            for i, block in enumerate(block_others):
                merged = sorted(B + block)
                other_remaining = [block_others[j] for j in range(len(block_others)) if j != i]
                result.append([A] + [merged] + other_remaining)
    return result

def display_priority_queue(pq):
    res = "[  "
    for acc_loss, partition in pq:
        res += f"({acc_loss:.4e}, {partition})  "
    res += "]"
    print(res)

def find_partitions_greedy_divisive(X, thresholds, priors, threshold_true, c, eps=1e-9, full_split=False, display_pq=False):
    n = len(priors)
    X_eps = np.arange(-1/c, 0, 1e-3).round(5)
    
    def block_cost(block):
        return (evaluate_partition(X, block, thresholds, priors, threshold_true, c)
                + evaluate_partition(X_eps, block, thresholds, priors, threshold_true, c) * eps)

    partition_0 = [list(range(n))]
    acc_loss_0 = block_cost(partition_0[0])
    pq = [(acc_loss_0, partition_0)]

    while pq:
        if display_pq:
            display_priority_queue(pq)
        acc_loss_merged, partition_merged = heapq.heappop(pq)
        if len(partition_merged[0])==1:
            return partition_merged
        partitions = split_partition(partition_merged, full_split)
        pq = []
        for partition in partitions:
            acc_loss_split = 0.
            for block in partition:
                acc_loss_split += block_cost(block) * np.sum(priors[block])
            gain = acc_loss_merged - acc_loss_split
            if  gain > 0:
                heapq.heappush(pq, (acc_loss_split, partition))
        if not pq:
            return partition_merged

In [341]:
X = np.arange(0., 1. + 1e-4, 1e-4).round(4)

In [354]:
priors = np.array([1., 0., 0., 0., 0.])
thresholds = np.array([0., 0.2, 0.4, 0.6, 0.8])
tt = 0.1
c = 0.75

# thresholds = np.array([0.0, 0.2, 0.6, 0.8, 1.0])
# priors = np.array([0.32, 0.02, 0.36, 0.3])
# c = 0.75
# tt = 0.1

# priors=np.array([0.15, 0.45, 0.15, 0.25])
# c=0.75
# tt=0.35

# priors= np.array([0.35, 0.25, 0.2, 0.15, 0.05])
# c=0.9
# tt=0.2

p_opt, p_gdy = find_partitions_optimal(X, thresholds, priors, tt, c), find_partitions_greedy_agglomerative(X, thresholds, priors, tt, c)
loss_opt = evaluate_system(X, p_opt, thresholds, priors, tt, c)
loss_gdy = evaluate_system(X, p_gdy, thresholds, priors, tt, c)
r = approximation_ratio(loss_opt, loss_gdy, "a")

print(f"OPT: {p_opt} ({loss_opt:.6f})")
print(f"GDY: {p_gdy} ({loss_gdy:.6f})")
print(f"r: {r:.6f}\n")

p_gdy_r = find_partitions_greedy_divisive(X, thresholds, priors, tt, c)
loss_gdy_r = evaluate_system(X, p_gdy_r, thresholds, priors, tt, c)
print(f"GDYR: {p_gdy_r} ({loss_gdy_r:.6f})")

p_gdy_rp = find_partitions_greedy_divisive(X, thresholds, priors, tt, c, full_split=True, display_pq=True)
loss_gdy_rp = evaluate_system(X, p_gdy_r, thresholds, priors, tt, c)
print(f"GDYRP: {p_gdy_rp} ({loss_gdy_rp:.6f})")

OPT: [[0, 1, 2, 3, 4]] (0.099990)
GDY: [[0, 1, 2, 3, 4]] (0.099990)
r: 0.000000

GDYR: [[0, 1, 2, 3, 4]] (0.099990)
[  (9.9990e-02, [[0, 1, 2, 3, 4]])  ]
GDYRP: [[0, 1, 2, 3, 4]] (0.099990)
